# 28 Local Model Refinement and Ablation

Local-only Phase 28 notebook. This notebook inspects baseline artifacts, runs controlled refinement experiments, runs required feature-group ablations, and writes local outputs.

## 1) Inputs and Baseline Artifacts

In [ ]:
from pathlib import Path
import json
import pandas as pd

from src.modeling.model_refinement import (
    read_baseline_artifacts,
    run_model_refinement,
    write_refinement_artifacts,
)

FEATURE_PATH = Path('local/derived/features/bluesky_engagement_features.parquet')
BASELINE_METRICS_PATH = Path('local/derived/modeling/baseline_model_metrics.json')
BASELINE_SUMMARY_PATH = Path('local/derived/modeling/baseline_model_summary.json')
BASELINE_CONFUSION_PATH = Path('local/derived/modeling/baseline_confusion_matrix.csv')
BASELINE_IMPORTANCE_PATH = Path('local/derived/modeling/baseline_feature_importance.parquet')

feature_df = pd.read_parquet(FEATURE_PATH)
baseline = read_baseline_artifacts(
    metrics_path=BASELINE_METRICS_PATH,
    summary_path=BASELINE_SUMMARY_PATH,
    confusion_path=BASELINE_CONFUSION_PATH,
    importance_path=BASELINE_IMPORTANCE_PATH,
)

print('Feature shape:', feature_df.shape)
print('Baseline best model:', baseline['metrics']['best_model_name'])
print('Baseline target:', baseline['metrics']['target_column'])


## 2) Run Controlled Refinement

In [ ]:
modeling_output = run_model_refinement(feature_df=feature_df, baseline_artifacts=baseline)

experiments_df = pd.DataFrame(modeling_output['refinement_experiments'])
display(experiments_df[[
    'experiment_name',
    'model_family',
    'balanced_resample',
    'accuracy',
    'balanced_accuracy',
    'macro_f1',
]].sort_values('experiment_name', kind='stable').reset_index(drop=True))

print('Selected refined experiment:', modeling_output['selected_refined_experiment_name'])
print('Recommendation:', modeling_output['comparison_to_baseline']['recommended_final_story'])


## 3) Feature-Group Ablation Results

In [ ]:
ablation_df = modeling_output['model_ablation_results']
display(ablation_df[[
    'experiment_name',
    'removed_feature_groups',
    'n_features_used',
    'skipped',
    'skipped_reason',
    'accuracy',
    'balanced_accuracy',
    'macro_f1',
]].sort_values('experiment_name', kind='stable').reset_index(drop=True))

print('Derived feature groups:')
for group_name, cols in modeling_output['feature_groups'].items():
    print(f'  {group_name}: {len(cols)}')


## 4) Confusion Matrix and Error Follow-Up

In [ ]:
comparison = modeling_output['comparison_to_baseline']
print('Metric deltas vs baseline:', comparison['metric_deltas'])

conf_df = modeling_output['refined_confusion_matrix']
display(conf_df)

pred_df = modeling_output['refined_prediction_sample']
display(pred_df[['uri', 'true_label', 'predicted_label', 'is_correct']])


## 5) Write Phase 28 Outputs

In [ ]:
artifact_paths = write_refinement_artifacts(
    modeling_output=modeling_output,
    output_dir='local/derived/modeling',
    sample_csv_path='data/samples/refined_predictions_sample_1000.csv',
)
artifact_paths
